In [ ]:
#模型預測
import pandas as pd
from xgboost import XGBClassifier

# ==========================================
# 1. 載入特徵資料 
# ==========================================
print("讀取資料")
final_features_df = pd.read_csv('(all)model_timeseries_data(20~22).csv')

# ==========================================
# 2. 準備特徵欄位 X (去掉 member_id 和 target)
# ==========================================
# 只要踢掉 member_id 跟 target，剩下特徵全部餵給模型！
X_2022 = final_features_df.drop(columns=['member_id', 'target'])

# ==========================================
# 3. 載入訓練好的模型
# ==========================================
print("載入模型...")
model = XGBClassifier()
model.load_model('time_xgb_model.json') # 注意這裡檔名要跟您存的一致！

# ==========================================
# 4. 進行預測
# ==========================================
print("進行預測中...")
# 將預測出來的 0, 1, 2 存入新欄位 'pred_label_code'
proba = model.predict_proba(X_2022)
t1, t2 = 0.42, 0.44  # 你訓練時找到的最佳門檻

pred_codes = []
for p in proba:
    if p[2] > t2:        
        pred_codes.append(2)
    elif p[1] > t1:      
        pred_codes.append(1)
    else:               
        pred_codes.append(0)

final_features_df['pred_label_code'] = pred_codes

# ==========================================
# 5. 載入目標繳交範本 (final.csv)
# ==========================================
print("合併並產出最終答案 final.xlsx...")
try:
    target_df = pd.read_csv('final.csv', encoding='utf-8-sig')
except:
    target_df = pd.read_csv('final.csv', encoding='cp950')

# ==========================================
# 6. 提取預測結果並改名對應
# ==========================================
pred_subset = final_features_df[['member_id', 'pred_label_code']]
pred_subset = pred_subset.rename(columns={
    'member_id': '新會員編號'
})

# ==========================================
# 7. 將預測結果合併到範本上
# ==========================================
# 以 target_df (老師給的範本) 的會員編號為主，填入預測結果
final_output = pd.merge(target_df[['新會員編號']], pred_subset, on='新會員編號', how='left')

# ==========================================
# 8. 格式美化：將 0, 1, 2 轉換回原本的文字標籤
# ==========================================
# 如果有缺漏值，預設給 0 (churn)，確保能成功轉換型態
final_output['pred_label_code'] = final_output['pred_label_code'].fillna(0).astype(int)

# 翻譯字典 (跟我們 LabelEncoder 的順序一致)
reverse_label_map = {0: 'churn', 1: 'partial churn', 2: 'loyal'}
final_output['label'] = final_output['pred_label_code'].map(reverse_label_map)

# 刪除過渡用的代碼欄位，只留下老師要的
final_output = final_output.drop(columns=['pred_label_code'])

# ==========================================
# 9. 輸出最終 Excel 檔案
# ==========================================
final_output.to_excel('final_test.xlsx', index=False)

print(f"🎉 答案檔案已儲存為 final_test.xlsx，共計 {len(final_output)} 筆會員資料。")


讀取資料
載入模型...
進行預測中...
合併並產出最終答案 final.xlsx...
🎉 答案檔案已儲存為 final_test.xlsx，共計 483 筆會員資料。


/var/folders/bc/3gcv0zzj4fgcc8gzvkdcr1940000gn/T/ipykernel_26228/2462329786.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_features_df['pred_label_code'] = pred_codes


In [61]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from tensorflow.keras.models import load_model
import json

# ==========================================
# 1. 載入特徵資料 
# ==========================================
print("讀取資料")
final_features_df = pd.read_csv('(all)model_timeseries_data(20~22).csv')

# ==========================================
# 2. 準備特徵欄位 X
# ==========================================
X_2022 = final_features_df.drop(columns=['member_id', 'target'])

# ==========================================
# 3. 載入 XGBoost 模型
# ==========================================
print("載入 XGBoost 模型...")
xgb_model = XGBClassifier()
xgb_model.load_model('xgb_model.json')

# ==========================================
# 4. 載入 LSTM 模型
# ==========================================
print("載入 LSTM 模型...")
lstm_model = load_model("lstm_model.keras")

# LSTM 需要正規化 + reshape
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_2022)
X_lstm = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))

# ==========================================
# 5. 載入 ensemble 權重
# ==========================================
try:
    weights = json.load(open("ensemble_weights.json"))
    w1, w2 = weights["xgb_weight"], weights["lstm_weight"]
except:
    w1, w2 = 1.0, 0.0   # 沒有 ensemble → 用純 XGBoost

print(f"Ensemble 權重：XGB={w1}, LSTM={w2}")

# ==========================================
# 6. 進行預測（兩個模型都要）
# ==========================================
print("進行預測中...")

xgb_probs = xgb_model.predict_proba(X_2022)
lstm_probs = lstm_model.predict(X_lstm)

# 加權平均
proba = w1 * xgb_probs + w2 * lstm_probs

# 門檻
t1, t2 = 0.36, 0.46

pred_codes = []
for p in proba:
    if p[2] > t2:
        pred_codes.append(2)
    elif p[1] > t1:
        pred_codes.append(1)
    else:
        pred_codes.append(0)

final_features_df['pred_label_code'] = pred_codes

# ==========================================
# 7. 合併 final.csv
# ==========================================
print("合併並產出最終答案 final.xlsx...")
try:
    target_df = pd.read_csv('final.csv', encoding='utf-8-sig')
except:
    target_df = pd.read_csv('final.csv', encoding='cp950')

pred_subset = final_features_df[['member_id', 'pred_label_code']]
pred_subset = pred_subset.rename(columns={'member_id': '新會員編號'})

final_output = pd.merge(target_df[['新會員編號']], pred_subset, on='新會員編號', how='left')

# 轉換 label
final_output['pred_label_code'] = final_output['pred_label_code'].fillna(0).astype(int)
reverse_label_map = {0: 'churn', 1: 'partial churn', 2: 'loyal'}
final_output['label'] = final_output['pred_label_code'].map(reverse_label_map)
final_output = final_output.drop(columns=['pred_label_code'])

# ==========================================
# 8. 輸出 Excel
# ==========================================
final_output.to_excel('final_test.xlsx', index=False)
print(f"🎉 答案檔案已儲存為 final_test.xlsx，共計 {len(final_output)} 筆會員資料。")


讀取資料
載入 XGBoost 模型...
載入 LSTM 模型...
Ensemble 權重：XGB=0.6, LSTM=0.4
進行預測中...
1810/1810 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
合併並產出最終答案 final.xlsx...
🎉 答案檔案已儲存為 final_test.xlsx，共計 483 筆會員資料。


/var/folders/bc/3gcv0zzj4fgcc8gzvkdcr1940000gn/T/ipykernel_26228/2471186957.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_features_df['pred_label_code'] = pred_codes


In [62]:
import pandas as pd
final = pd.read_excel('final_test.xlsx')
print("預測分布：")
print(final['label'].value_counts())
print(f"\n比例：")
print(final['label'].value_counts(normalize=True))

預測分布：
label
churn            202
partial churn    159
loyal            122
Name: count, dtype: int64

比例：
label
churn            0.418219
partial churn    0.329193
loyal            0.252588
Name: proportion, dtype: float64
